# 06. Model comparison

All models side by side on the same validation rows and metrics, the final model choice, and the single score on the test set.

## Weighted validation sample: check before use

The LoRA LLM, and possibly TabPFN, are too slow to score all 965,524 validation rows, so they may be scored on all 1,036 laundering rows plus a random 200,000 normal rows, each weighted to stand for 4.8 rows. Before relying on that, the weighted estimate is compared with full validation for the three models already scored in full, over 5 random samples. The rule, fixed before seeing the results: if any weighted PR-AUC differs from the full value by more than 0.01, the slow models are scored on all of validation instead.

In [1]:
# Reload what this section needs from disk, so it runs without the training cells above.
from pathlib import Path

import mlflow
import mlflow.lightgbm
import numpy as np
import pandas as pd

from bankguard.data import load_accounts, load_transactions, split_by_date
from bankguard.evaluate import evaluate, weighted_sample
from bankguard.features import FEATURES, build_features, fit_currency_median
from bankguard.model import predict

DATA = Path("../data")
tx = load_transactions(DATA / "HI-Small_Trans.csv")
accounts = load_accounts(DATA / "HI-Small_accounts.csv")
splits = split_by_date(tx)
train, val = splits["train"], splits["validation"]
X = build_features(tx, accounts, fit_currency_median(train))
X_val, y_val = X.loc[val.index, FEATURES], val["is_laundering"].to_numpy()
del tx, X

mlflow.set_tracking_uri("sqlite:///../mlflow.db")
runs = mlflow.search_runs(
    experiment_names=["bankguard-laundering"],
    filter_string="tags.mlflow.runName = 'LightGBM, owner attributes'",
)
run_id = runs.loc[0, "run_id"]
lgbm = mlflow.lightgbm.load_model(f"runs:/{run_id}/model")
pre = mlflow.artifacts.load_dict(f"runs:/{run_id}/preprocessing.json")

scores = {
    "LightGBM": predict(lgbm, X_val, pre["features"], pre["categories"]),
    "DistilBERT": np.load("../models/distilbert/val_scores.npy"),
    "ModernBERT": np.load("../models/modernbert/modernbert_val_scores.npy"),
}
np.save("../models/lightgbm_val_scores.npy", scores["LightGBM"])
print({name: round(evaluate(y_val, s, name)["pr_auc"], 4) for name, s in scores.items()})

c:\SAP\Projects\Bankguard\bank-guard\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'LightGBM': 0.3412, 'DistilBERT': 0.2281, 'ModernBERT': 0.3023}


In [2]:
full = {name: evaluate(y_val, s, name)["pr_auc"] for name, s in scores.items()}

rows = []
for seed in range(5):
    idx, w = weighted_sample(y_val, n_negatives=200_000, seed=seed)
    for name, s in scores.items():
        estimate = evaluate(y_val[idx], s[idx], name, sample_weight=w)["pr_auc"]
        rows.append({"seed": seed, "model": name, "difference": estimate - full[name]})

check = pd.DataFrame(rows)
print(f"largest difference: {check['difference'].abs().max():.4f} (rule: at most 0.0100)")
check.pivot(index="seed", columns="model", values="difference").round(4)

largest difference: 0.0107 (rule: at most 0.0100)


model,DistilBERT,LightGBM,ModernBERT
seed,,,
0,-0.0009,0.0103,0.0004
1,-0.0075,0.0087,-0.0067
2,0.0037,-0.0014,0.0075
3,-0.0026,0.0050,-0.0041
4,-0.0040,0.0065,0.0107


Over 5 random samples, the weighted PR-AUC differed from the full value by at most 0.0107 (ModernBERT, seed 4; LightGBM reached 0.0103 on seed 0), just over the 0.01 limit fixed in advance. Following the rule, models too slow for a single run are scored on all of validation instead, split across separate runs if needed. The differences are small compared with the gaps between models (0.04 to 0.11 PR-AUC), so the weighted sample would not have changed their ranking, but the rule was set before the check and is kept.

## All models on validation

Every model scored on the same 965,524 validation rows with the same metrics, from the two one-line rules up to TabPFN. Feature ablations of LightGBM are in `02_baselines.ipynb`. The Qwen LoRA model is added when its scoring run finishes.

In [3]:
scores["ModernBERT, 2 epochs"] = scores.pop("ModernBERT")
scores["ModernBERT, 4 epochs"] = np.load("../models/modernbert/modernbert_4ep_val_scores.npy")
scores["TabPFN-3.5-Fast"] = np.load("../models/tabpfn/tabpfn_val_scores.npy")

rules = {
    "Rule: every ACH payment": (val["payment_format"] == "ACH").to_numpy().astype(float),
    "Rule: every new counterparty": (
        (X_val["pair_prior_3d"] == 0) & (X_val["self_transfer"] == 0)
    ).to_numpy().astype(float),
}

table = pd.DataFrame([evaluate(y_val, s, name) for name, s in {**rules, **scores}.items()])
table.sort_values("pr_auc").reset_index(drop=True).round(4)

,model,pr_auc,recall_at_0.1%_fpr,recall_at_1.0%_fpr
0,Rule: every ACH payment,0.0067,0.0000,0.0000
1,Rule: every new counterparty,0.0156,0.0000,0.0000
2,DistilBERT,0.2281,0.2963,0.7259
3,"ModernBERT, 2 epochs",0.3023,0.3600,0.7828
4,LightGBM,0.3412,0.3793,0.7635
5,"ModernBERT, 4 epochs",0.3526,0.3755,0.7606
6,TabPFN-3.5-Fast,0.3985,0.3967,0.7963


The two one-line rules reproduce `02_baselines.ipynb` (PR-AUC 0.0067 and 0.0156). They cannot rank transactions, only flag or not, and each flags far more than 1% of normal transactions, so neither fits within a 0.1% or 1% alert budget. Every trained model is 15 to 60 times better on PR-AUC. TabPFN-3.5-Fast leads on all three metrics. ModernBERT after four epochs and LightGBM are close behind and level within LightGBM's spread across seeds, with ModernBERT slightly ahead on PR-AUC and LightGBM slightly ahead on both recalls. ModernBERT after two epochs has the highest recall at a 1% budget of the models below TabPFN, but ranks the very top less sharply.

In [4]:
from bankguard.data import load_patterns, typology_for
from bankguard.evaluate import laundering_group, recall_by_group

patterns = load_patterns(DATA / "HI-Small_Patterns.txt")
val_group = laundering_group(typology_for(val, patterns), X_val["sender_out_3d"])

group_table = pd.concat(
    {name: recall_by_group(y_val, s, val_group)["recall_at_1.0%_fpr"] for name, s in scores.items()},
    axis=1,
)
group_table.insert(0, "laundering", recall_by_group(y_val, scores["LightGBM"], val_group)["laundering"])
group_table.round(3)

,laundering,LightGBM,DistilBERT,"ModernBERT, 2 epochs","ModernBERT, 4 epochs",TabPFN-3.5-Fast
patterned,657,0.913,0.854,0.930,0.896,0.945
"unpatterned, quiet sender",246,0.764,0.772,0.813,0.793,0.829
"unpatterned, hub sender",133,0.023,0.008,0.000,0.030,0.000


At a 1% false positive rate, TabPFN leads on both groups that can be detected: patterned laundering (94.5% against 91.3% for LightGBM) and unpatterned laundering from quiet senders (82.9% against 76.4%, about 16 more of 246 cases). Laundering from hub senders is missed by every model, between 0% and 3% recall, across gradient boosting, two fine-tuned text transformers and a tabular foundation model. That five different learners fail on the same group is strong evidence that the 22 features carry no signal for it. With hub sent laundering undetected, recall cannot exceed (657 + 246) / 1,036 = 87.2%; TabPFN's 79.6% is most of the way there, so further gains need new information about hub senders, such as graph structure beyond trailing counts, rather than a different model.